# JobRec – AI-Powered Job Recommendation System
### Using Machine Learning for Skills-Based Career Matching
**Syed Mohiuddin Mudassir (415789) · MSc Computer Science · Supervisor: Dr. Paul Sant**

Run the cells **top to bottom** (Runtime ▸ Run all also works – the first cell will ask you to upload `jobrec.zip`).

| Step | What happens |
|---|---|
| 1 | upload & unzip the project |
| 2 | install requirements |
| 3 | look at the dataset |
| 4 | train the hybrid recommender |
| 5 | run the offline evaluation (precision@k / recall@k) |
| 6 | try the recommender directly in Python |
| 7 | launch the Flask web app and open it in the browser |

## 1 · Upload the project zip

In [ ]:
import os, zipfile
from google.colab import files
if not os.path.exists('jobrec'):
    print('Please choose jobrec.zip from your computer ...')
    up = files.upload()
    name = [n for n in up if n.endswith('.zip')][0]
    with zipfile.ZipFile(name) as z:
        z.extractall('.')
%cd jobrec
print('Project folder:', os.getcwd())
print(os.listdir('.'))

## 2 · Install requirements

In [ ]:
!pip install -q -r requirements.txt
import pandas, sklearn, flask
print('pandas', pandas.__version__, '| scikit-learn', sklearn.__version__, '| Flask', flask.__version__)

## 3 · The dataset
8,000 real postings from the public Kaggle dataset *Data Analyst Job Postings [Pay, Skills, Benefits]* (Luke Barousse).

In [ ]:
import pandas as pd
from jobrec import load_jobs
jobs = load_jobs()
print('postings:', len(jobs), '| distinct skills:', jobs['skill_list'].explode().nunique())
display(jobs[['job_id','title','company','location','work_from_home','salary_yearly','skills']].head(8))
print('\nTop 15 skills:')
display(jobs['skill_list'].explode().value_counts().head(15).to_frame('postings'))

In [ ]:
import matplotlib.pyplot as plt
top = jobs['skill_list'].explode().value_counts().head(20)
plt.figure(figsize=(10,4)); top.plot(kind='bar'); plt.title('Most requested skills'); plt.ylabel('postings'); plt.tight_layout(); plt.show()

## 4 · Train the hybrid recommender
Content-based (TF-IDF + cosine + skill overlap) **and** collaborative filtering (item-kNN + TruncatedSVD), trained on 300 mock users / ~4,200 interactions.

In [ ]:
!python train.py

## 5 · Offline evaluation – precision@k, recall@k, hit-rate@k, NDCG@k
70 % of every user's interactions are used for training, 30 % are held out. A recommendation is a *hit* if it is in the held-out set.

In [ ]:
!python evaluate.py
from IPython.display import Image, display
display(Image('outputs/precision_recall_at_k.png'))

## 6 · Try the recommender directly

In [ ]:
from jobrec import HybridRecommender
rec = HybridRecommender.load()

my_skills = 'Python, SQL, Power BI, Excel, machine learning'
rows, mode = rec.recommend(my_skills,
                           experience='MSc Computer Science graduate, built ML models and a Flask app',
                           desired_title='Data Scientist', k=10)
print('mode:', mode)
pd.DataFrame(rows)[['title','company','location','score','matched_skills','missing_skills']]

In [ ]:
# with filters: remote only, minimum salary
rows, mode = rec.recommend(my_skills, desired_title='Data Analyst', k=5, remote_only=True, min_salary=70000)
pd.DataFrame(rows)[['title','company','location','work_from_home','salary_yearly','score','matched_skills']]

In [ ]:
# collaborative part: what would a known mock user (id 7) get?  ->  'hybrid' mode
import json
mock = json.load(open('models/mock_users.json'))
u = mock['users'][6]
print('user:', u['name'], '| skills:', u['skills'], '| wants:', u['desired_title'])
rows, mode = rec.recommend(u['skills'], u['experience'], u['desired_title'], user_id=u['user_id'], k=5)
print('mode:', mode)
pd.DataFrame(rows)[['title','company','score','content_score','cf_score','matched_skills']]

## 7 · Launch the web application
The app starts in a background thread and Colab gives us a private URL to it. Log in with **demo / demo123**.

If the link does not open, run the *alternative* cell below it (uses ngrok – needs a free auth token from https://dashboard.ngrok.com).

In [ ]:
import threading, time, sys, os
sys.path.insert(0, os.getcwd())
os.environ['PORT'] = '5000'
from app.app import app as flask_app

def run():
    flask_app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False)

if not any(t.name == 'jobrec-server' for t in threading.enumerate()):
    threading.Thread(target=run, name='jobrec-server', daemon=True).start()
    time.sleep(3)

from google.colab.output import eval_js
url = eval_js('google.colab.kernel.proxyPort(5000)')
from IPython.display import HTML
display(HTML(f'<h3>✅ JobRec is running: <a href="{url}" target="_blank">{url}</a></h3><p>Login: <b>demo</b> / <b>demo123</b></p>'))

In [ ]:
# ALTERNATIVE (only if the link above does not work): public URL via ngrok
# 1) create a free account at https://dashboard.ngrok.com and copy your auth token
# 2) paste it below and run this cell
NGROK_TOKEN = ''
if NGROK_TOKEN:
    !pip install -q pyngrok
    from pyngrok import ngrok
    ngrok.set_auth_token(NGROK_TOKEN)
    public_url = ngrok.connect(5000).public_url
    print('Public URL:', public_url, '  (login demo / demo123)')

In [ ]:
# Optional: show the app inside the notebook
from IPython.display import IFrame
IFrame(src=url, width='100%', height=700)

## 8 · REST API check

In [ ]:
import requests
r = requests.get('http://127.0.0.1:5000/api/recommend', params={'skills':'python,sql,tableau','title':'Data Analyst','k':3}).json()
print('mode:', r['mode'])
for x in r['recommendations']:
    print(f"{x['score']:.2f}  {x['title']} @ {x['company']}  matched={x['matched_skills']}")